# 04 — Stacking Ensemble (+ within-fold SMOTE)

_Placeholder — RF + XGBoost base learners -> Logistic Regression meta-learner, with SMOTE applied strictly within each CV fold (core novelty)._

In [1]:
import os, time, warnings
import numpy as np
import joblib
from collections import Counter
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.metrics import classification_report, accuracy_score, f1_score

warnings.filterwarnings("ignore")   # silence the harmless XGBoost GPU predict notice

d = joblib.load(os.path.join(os.getcwd(), "..", "data", "processed", "preprocessed.joblib"))
X_train, X_test = d["X_train"], d["X_test"]
y_train, y_test = d["y_train"], d["y_test"]
class_names = d["label_classes"]
print("Loaded. X_train:", X_train.shape, " classes:", len(class_names))

def report(name, y_true, y_pred):
    print(f"\n===== {name} =====")
    print("Accuracy    :", round(accuracy_score(y_true, y_pred), 4))
    print("Macro-F1    :", round(f1_score(y_true, y_pred, average='macro'), 4))
    print("Weighted-F1 :", round(f1_score(y_true, y_pred, average='weighted'), 4))
    print(classification_report(y_true, y_pred, target_names=class_names, digits=3, zero_division=0))

Loaded. X_train: (2262263, 64)  classes: 11


In [2]:
# SMOTE target rule: oversample every class with fewer than `floor` samples UP TO `floor`.
# Big classes (BENIGN, DoS Hulk, PortScan, DDoS) are left untouched. A callable adapts
# to each fold's own class counts — which is what keeps SMOTE strictly within the fold.
def make_smote_floor(floor):
    def f(y):
        counts = Counter(y)
        return {c: floor for c, n in counts.items() if n < floor}
    return f

def build_stack(floor=50_000):
    rf_pipe = ImbPipeline([
        ("smote", SMOTE(sampling_strategy=make_smote_floor(floor), k_neighbors=5, random_state=42)),
        ("rf", RandomForestClassifier(n_estimators=100, max_depth=30,
                                      min_samples_leaf=2, n_jobs=4, random_state=42)),
    ])
    xgb_pipe = ImbPipeline([
        ("smote", SMOTE(sampling_strategy=make_smote_floor(floor), k_neighbors=5, random_state=42)),
        ("xgb", XGBClassifier(n_estimators=100, max_depth=8, learning_rate=0.3,
                              tree_method="hist", device="cuda",
                              eval_metric="mlogloss", random_state=42)),
    ])
    return StackingClassifier(
        estimators=[("rf", rf_pipe), ("xgb", xgb_pipe)],
        final_estimator=LogisticRegression(max_iter=1000),
        stack_method="predict_proba",
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
        n_jobs=1,            # sequential — keeps memory bounded on 16 GB (RF parallelises internally)
        passthrough=False,
    )

print("Stacking builder ready.")

Stacking builder ready.


In [3]:
# Confirms the whole pipeline executes end-to-end before we commit to the long full run.
# Trained on a tiny subsample, so these numbers are NOT a real result — we only care that it runs.
Xs, _, ys, _ = train_test_split(X_train, y_train, train_size=40000,
                                stratify=y_train, random_state=42)
t0 = time.time()
dry = build_stack(floor=5000)     # small floor so the dry run is quick
dry.fit(Xs, ys)
print(f"Dry run fit OK in {time.time()-t0:.1f}s")
report("Stacking DRY RUN (subsample — ignore the scores)", y_test, dry.predict(X_test))

Dry run fit OK in 127.5s

===== Stacking DRY RUN (subsample — ignore the scores) =====
Accuracy    : 0.9978
Macro-F1    : 0.9434
Weighted-F1 : 0.9977
                  precision    recall  f1-score   support

          BENIGN      0.999     0.999     0.999    454264
             Bot      0.710     0.583     0.640       391
            DDoS      0.999     0.998     0.998     25605
   DoS GoldenEye      0.995     0.976     0.985      2059
        DoS Hulk      0.997     0.997     0.997     46025
DoS Slowhttptest      0.899     0.943     0.920      1100
   DoS slowloris      0.995     0.953     0.973      1159
     FTP-Patator      0.994     0.995     0.994      1587
        PortScan      0.994     0.999     0.996     31761
     SSH-Patator      0.999     0.989     0.994      1179
      Web Attack      0.882     0.878     0.880       436

        accuracy                          0.998    565566
       macro avg      0.951     0.937     0.943    565566
    weighted avg      0.998     0.99

In [4]:
t0 = time.time()
stack = build_stack(floor=50_000)      # oversample small/mid classes to 50k within each fold
stack.fit(X_train, y_train)
print(f"Full stacking model trained in {(time.time()-t0)/60:.1f} min")

stack_pred = stack.predict(X_test)
report("Stacking Ensemble (within-fold SMOTE) — FULL", y_test, stack_pred)

# Save the fitted model for the FastAPI backend later
out = os.path.join(os.getcwd(), "..", "models", "stacking_model.joblib")
joblib.dump(stack, out, compress=3)
print("Saved model to", out)

Full stacking model trained in 51.3 min

===== Stacking Ensemble (within-fold SMOTE) — FULL =====
Accuracy    : 0.999
Macro-F1    : 0.9755
Weighted-F1 : 0.999
                  precision    recall  f1-score   support

          BENIGN      1.000     0.999     0.999    454264
             Bot      0.752     0.844     0.795       391
            DDoS      1.000     1.000     1.000     25605
   DoS GoldenEye      0.998     0.995     0.996      2059
        DoS Hulk      0.999     0.999     0.999     46025
DoS Slowhttptest      0.950     0.991     0.970      1100
   DoS slowloris      0.997     0.998     0.997      1159
     FTP-Patator      0.999     0.998     0.999      1587
        PortScan      0.994     0.999     0.997     31761
     SSH-Patator      1.000     1.000     1.000      1179
      Web Attack      0.982     0.975     0.978       436

        accuracy                          0.999    565566
       macro avg      0.970     0.982     0.976    565566
    weighted avg      0.999

PicklingError: Can't pickle <function make_smote_floor.<locals>.f at 0x000002B278580CC0>: it's not found as __main__.make_smote_floor.<locals>.f

In [5]:
import os, joblib
from imblearn.over_sampling import SMOTE

# Swap the un-picklable SMOTE floor rule out of the fitted model.
# Safe: samplers only act during training, never during prediction.
def clear_smote(pipe):
    for _, step in pipe.steps:
        if isinstance(step, SMOTE):
            step.sampling_strategy = "auto"

for _, est in stack.estimators:    # original pipeline templates
    clear_smote(est)
for est in stack.estimators_:      # fitted pipelines
    clear_smote(est)

out = os.path.join(os.getcwd(), "..", "models", "stacking_model.joblib")
joblib.dump(stack, out, compress=3)
print("Saved model to", out)

# sanity check: reload and confirm it still predicts
reloaded = joblib.load(out)
print("Reload OK — sample preds:", reloaded.predict(X_test[:5]))

Saved model to d:\Capstone Project\nids-capstone\notebooks\..\models\stacking_model.joblib
Reload OK — sample preds: [0 0 0 0 0]


In [6]:
import glob, os
import numpy as np, pandas as pd, joblib

DATA_DIR = r"D:\Capstone Project\MachineLearningCVE"
d = joblib.load(os.path.join(os.getcwd(), "..", "data", "processed", "preprocessed.joblib"))
feature_names = d["feature_names"]

frames = []
for f in sorted(glob.glob(os.path.join(DATA_DIR, "*.csv"))):
    df = pd.read_csv(f, low_memory=False)
    df.columns = df.columns.str.strip()
    frames.append(df)
data = pd.concat(frames, ignore_index=True)

# same label handling as training
data["Label"] = data["Label"].str.strip()
data.loc[data["Label"].str.startswith("Web Attack"), "Label"] = "Web Attack"
data = data[~data["Label"].isin(["Heartbleed", "Infiltration"])]

# keep the 64 model features (RAW, unscaled) + Label; drop inf/NaN rows
data = data[feature_names + ["Label"]].replace([np.inf, -np.inf], np.nan).dropna()

# balanced demo stream: up to 150 flows per class, then shuffle
sample = (data.groupby("Label", group_keys=False)
              .apply(lambda g: g.sample(min(len(g), 150), random_state=42))
              .sample(frac=1, random_state=42).reset_index(drop=True))

out = os.path.join(os.getcwd(), "..", "data", "processed", "replay_sample.csv")
sample.to_csv(out, index=False)
print("Saved", len(sample), "flows to", out)
print(sample["Label"].value_counts())

Saved 1650 flows to d:\Capstone Project\nids-capstone\notebooks\..\data\processed\replay_sample.csv
Label
DoS GoldenEye       150
BENIGN              150
DoS slowloris       150
FTP-Patator         150
Bot                 150
PortScan            150
Web Attack          150
DDoS                150
DoS Hulk            150
SSH-Patator         150
DoS Slowhttptest    150
Name: count, dtype: int64
